# deep-memory-agent — LongMemEval-style benchmark

Measures the whole memory pipeline end to end, not a retriever over a frozen log.
Each case gets its own memory tree, its own agents and its own simulated clock:

```
<EXPERIMENT_ROOT>/<question_id>/memory/       the tree this case built
<EXPERIMENT_ROOT>/<question_id>/result.json   what this case produced
<EXPERIMENT_ROOT>/result.json                 config + every case + the summary
```

The run is resumable — a case that already has a `result.json` is not re-run — so
an interrupted run picks up where it stopped instead of paying twice.

**Before running the `large` scale, read the cost estimate cell.** Ingestion is one
agent invocation per session, and the `large` scale is ~48 sessions per case.

## 1. Configuration

Everything that defines a run. Nothing below this cell needs editing.

In [ ]:
from pathlib import Path

# --- where things live -------------------------------------------------------
BENCHMARK_DIR = (
    Path("benchmark").resolve() if Path("benchmark").exists() else Path.cwd()
)
LONGMEMEVAL_DIR = BENCHMARK_DIR / "longmemeval" / "data"
OPERATIONAL_DIR = BENCHMARK_DIR / "data"
EXPERIMENT_ROOT = BENCHMARK_DIR / "runs" / "small-cold"

# --- what to run -------------------------------------------------------------
DATASET = "longmemeval"  # "longmemeval" | "operational"
SCALE = "small"  # "small" (oracle, ~2 sessions) | "large" (s, ~48 sessions)
N_PER_CATEGORY = 3  # cases sampled per category
CATEGORIES = None  # None = every category the corpus can supply
SEED = 0  # makes the sample reproducible

# --- how the memory is built -------------------------------------------------
CONSOLIDATION_MODE = "none"  # "none" | "periodic" | "final" | "periodic+final"
CONSOLIDATE_EVERY_N = 10  # sessions between periodic consolidations

# --- how it is graded --------------------------------------------------------
RECALL_THRESHOLD = 1.0  # 1.0 is LongMemEval's strict reading
CHAIN_OF_NOTE = True  # extract-then-reason answering prompt

# --- execution ---------------------------------------------------------------
MAX_WORKERS = 4  # cases graded concurrently; they share nothing
RECURSION_LIMIT = 60  # cap on agent steps per invocation
RESUME = True  # skip cases that already have a result.json

# --- models ------------------------------------------------------------------
# Any OpenAI-compatible endpoint: a local server (vLLM, LM Studio, llama.cpp),
# OpenRouter, or OpenAI itself. The judge should be at least as strong as the
# model under test — in the paper the judge is GPT-4o grading everything else.
AGENT_MODEL = "gpt-4o-mini"
JUDGE_MODEL = "gpt-4o"
BASE_URL = None  # e.g. "http://localhost:4321/v1"; None = OpenAI
API_KEY = None  # None reads OPENAI_API_KEY from the environment
TEMPERATURE = 0.0
TIMEOUT = 240

## 2. Models

Factories, not instances: each case builds its own clients so nothing leaks between cases running side by side.

In [ ]:
import os
import sys

if str(BENCHMARK_DIR) not in sys.path:
    sys.path.insert(0, str(BENCHMARK_DIR))

from langchain_openai import ChatOpenAI

from dma_bench.runner import Runtime


def _model(name: str) -> ChatOpenAI:
    return ChatOpenAI(
        model=name,
        base_url=BASE_URL,
        api_key=API_KEY or os.environ.get("OPENAI_API_KEY", "not-needed"),
        temperature=TEMPERATURE,
        timeout=TIMEOUT,
        max_retries=2,
    )


runtime = Runtime(
    agent_model=lambda: _model(AGENT_MODEL),
    judge_model=lambda: _model(JUDGE_MODEL),
)
print(f"agent: {AGENT_MODEL}   judge: {JUDGE_MODEL}   endpoint: {BASE_URL or 'openai'}")

## 3. Load the cases

Sampling is balanced per category and reproducible for a given `SEED`.

In [ ]:
from collections import Counter

from dma_bench.datasets import load_dataset
from dma_bench.schema import RunConfig

data_root = LONGMEMEVAL_DIR if DATASET == "longmemeval" else OPERATIONAL_DIR
cases = load_dataset(
    DATASET,
    data_root,
    scale=SCALE,
    categories=CATEGORIES,
    n_per_category=N_PER_CATEGORY,
    seed=SEED,
)

config = RunConfig(
    experiment_root=str(EXPERIMENT_ROOT),
    dataset=DATASET,
    scale=SCALE,
    n_per_category=N_PER_CATEGORY,
    consolidation_mode=CONSOLIDATION_MODE,
    consolidate_every_n=CONSOLIDATE_EVERY_N,
    recall_threshold=RECALL_THRESHOLD,
    chain_of_note=CHAIN_OF_NOTE,
    agent_model=AGENT_MODEL,
    judge_model=JUDGE_MODEL,
    max_workers=MAX_WORKERS,
    recursion_limit=RECURSION_LIMIT,
    seed=SEED,
    resume=RESUME,
)

print(f"{len(cases)} cases from {DATASET}/{SCALE}")
for name, count in sorted(Counter(case.category.value for case in cases).items()):
    print(f"  {name:22s} {count}")

## 4. What this will cost

Ingestion dominates: one agent invocation per session, each several model calls
once tool use is counted. Read this before running the `large` scale.

In [ ]:
from dma_bench.runner import estimate_run, iter_pending

estimate = estimate_run(cases, config)
pending = iter_pending(cases, config)

for key, value in estimate.items():
    print(f"{key:32s} {value:,}")
print(
    f"\n{len(pending)} of {len(cases)} cases still to run "
    f"({len(cases) - len(pending)} already have a result.json)"
)

## 5. Run

Each finished case is written to disk as it completes, so stopping here loses nothing.

In [ ]:
import itertools
import time

from dma_bench.runner import run_experiment

started = time.monotonic()
counter = itertools.count(1)


def progress(result):
    recall = result.retrieval.recall
    status = (
        "ERROR"
        if result.error
        else (
            f"qa={'Y' if result.qa.correct else 'n'} "
            f"recall={recall if recall is not None else '-'} "
            f"cons={'Y' if result.consolidation.correct else 'n'}"
        )
    )
    print(
        f"[{next(counter):3d}/{len(cases)}] {result.question_id:24s} "
        f"{result.category.value:22s} {status}  "
        f"({time.monotonic() - started:.0f}s)"
    )


experiment = run_experiment(cases, config, runtime, on_result=progress)
print(f"\nwrote {EXPERIMENT_ROOT / 'result.json'}")

## 6. Results

In [ ]:
import pandas as pd

from dma_bench.report import category_table, cost_table, decomposition_table

pd.set_option("display.width", 140)
summary = experiment.summary

print(
    f"cases {summary['cases']}  failed {summary['failed_cases']}  "
    f"duration {summary.get('duration_s', 0)}s\n"
)
category_table(summary).round(3)

### The three-stage decomposition

LongMemEval decomposes failures across two stages because it has two. A system
that writes its own memory has three, so this is eight cells rather than four —
and each cell names a different file to go and change.

In [ ]:
decomposition_table(summary)

In [ ]:
cost_table(summary)

## 7. Charts

In [ ]:
%matplotlib inline
from dma_bench.report import (
    plot_accuracy_by_category,
    plot_decomposition,
    plot_recall_vs_accuracy,
)

plot_accuracy_by_category(summary, title=f"{DATASET}/{SCALE} — {CONSOLIDATION_MODE}");

In [ ]:
plot_decomposition(summary);

In [ ]:
plot_recall_vs_accuracy(experiment.results);

## 8. Consolidation ablation

`consolidation-quality` is not a question category — it is the difference between
two runs over the same cases, one with consolidation and one without. Run this
after the cell above, with a second `EXPERIMENT_ROOT` so both stay on disk.

Skip it on the `small` scale: with two sessions per case there is nothing for a
consolidation pass to consolidate.

In [ ]:
ABLATION_MODES = ["none", "periodic+final"]

summaries = {}
for mode in ABLATION_MODES:
    arm_config = config.model_copy(
        update={
            "consolidation_mode": mode,
            "experiment_root": str(
                EXPERIMENT_ROOT.parent
                / f"{EXPERIMENT_ROOT.name}-{mode.replace('+', '-')}"
            ),
        }
    )
    print(f"\n=== arm: {mode} ===")
    arm = run_experiment(cases, arm_config, runtime, on_result=progress)
    summaries[mode] = arm.summary

pd.DataFrame(
    {
        mode: {
            "QA accuracy": values.get("qa_accuracy"),
            "Retrieval correct": values.get("retrieval_correct_rate"),
            "Retrieval recall": values.get("retrieval_recall"),
            "Consolidation correct": values.get("consolidation_correct_rate"),
            "Active entries": values.get("cost", {}).get("active_entries"),
            "Superseded entries": values.get("cost", {}).get("superseded_entries"),
        }
        for mode, values in summaries.items()
    }
).round(3)

In [ ]:
from dma_bench.report import plot_ablation

plot_ablation(summaries, title="Effect of consolidation on QA accuracy");

## 9. Reload a finished run

The results are complete on disk, so a report can be rebuilt without spending
anything. Point this at any experiment root.

In [ ]:
from dma_bench.runner import load_experiment

reloaded = load_experiment(EXPERIMENT_ROOT)
category_table(reloaded.summary).round(3)

# If a run was interrupted before result.json was written, the per-case files
# are still complete on their own:
#
#     from dma_bench.metrics import summarise
#     from dma_bench.runner import load_case_results
#
#     category_table(summarise(load_case_results(EXPERIMENT_ROOT))).round(3)